In [1]:
import numpy as np
import torch
import copy
import os
from torch.utils.data import DataLoader
from torch.nn.functional import cross_entropy
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from transformers import get_scheduler # type: ignore
from tqdm import tqdm

def set_seed(seed):
    # Set seed for torch
    torch.manual_seed(seed)
    
    # If using CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    # Set seed for NumPy
    np.random.seed(seed)
    # Set deterministic behavior for CUDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set PYTHONHASHSEED
    os.environ['PYTHONHASHSEED'] = str(seed)
    
# Set seed
set_seed(42)
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"


# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

# The dataset["train"] subset has 25,000 samples. We'll split it into two halves.
full_train_data = dataset["train"]
# cut the dataset to 1000 samples for testing
full_train_data = full_train_data.select(range(500))

# Shuffle the training set for a fair split
full_train_data = full_train_data.shuffle(seed=42)

# Split into IN set (first half) and OUT set (second half)
canary_frac = 0.5
n_canaries = int(len(full_train_data) * canary_frac)
canaries = full_train_data.select(range(0, n_canaries))
non_canaries = full_train_data.select(range(n_canaries, len(full_train_data)))
scores = np.zeros(n_canaries)

# We will train on in_set, and out_set will be used only for membership inference evaluation
test_data = dataset["test"]  # Standard test set (25,000 samples) for normal model evaluation if desired

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
canaries = canaries.map(tokenize_function, batched=True)
non_canaries = non_canaries.map(tokenize_function, batched=True)

# (Optional) Tokenize standard test set
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
non_canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# subsample canaries & make new dataloader
true_in_out = torch.distributions.bernoulli.Bernoulli(torch.ones(n_canaries) * 0.5).sample()
true_in_out = true_in_out.numpy()
canaries_in_idx = torch.nonzero(torch.tensor(true_in_out))

# concatenate non_canaries data with samples from canaries with canaries_in_idx
subsampled_train_data = concatenate_datasets([non_canaries, canaries.select(canaries_in_idx)])
subsampled_train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

training_args = TrainingArguments(
            output_dir="./distilbert-imdb",
            overwrite_output_dir=True,
            num_train_epochs=1, # Set desired number of epochs - Commented out to use max_steps
            # max_steps=100,  # Set desired number of training steps
            per_device_train_batch_size=16, # set automatically during training
            per_device_eval_batch_size=16,
            learning_rate=5e-5,             
            weight_decay=0.0,               
            adam_beta1=0.9,                   
            adam_beta2=0.999,                 
            adam_epsilon=1e-8,                
            eval_strategy="no", #"epoch",
            save_strategy="no",  #"epoch", Disable saving the model during training
            logging_dir="./logs",
            logging_steps=100,
            seed=42
        )

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# def train_llm_with_opacus(model, device, train_loader, optimizer, scheduler, privacy_engine, training_args, sigma, client_id=None): 


# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

def train_llm_with_opacus(model, device, train_data, training_args, sigma, delta, client_id=None): 
    # Create the dataloader
    train_loader = DataLoader(
        train_data,
        batch_size=training_args.per_device_train_batch_size,
        shuffle=True,
    )
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=training_args.learning_rate,
        betas=(training_args.adam_beta1, training_args.adam_beta2),
        eps=training_args.adam_epsilon,
        weight_decay=training_args.weight_decay,
    )
    scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * training_args.num_train_epochs,
    )
        
    # Freeze the word and position embeddings - few params (cannot be optimized with opacus)
    model.train()
    model.distilbert.embeddings.position_embeddings.weight.requires_grad = False

    # Opacus privacy engine
    privacy_engine = opacus.PrivacyEngine(accountant='rdp', secure_mode=False)
    model, optimizer, train_loader = privacy_engine.make_private(
                        module=model,
                        optimizer=optimizer,
                        data_loader=train_loader,
                        noise_multiplier=sigma,
                        max_grad_norm=sensitivity,
                        poisson_sampling=False,
                    )

    # 4) Training loop
    training_loss_history = []
    for epoch in range(training_args.num_train_epochs):
        # print(f"Epoch {epoch + 1}/{epochs}")
        progress_bar = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            bar_format="{percentage:3.0f}%|{bar:10}| {n_fmt}/{total_fmt} - {postfix} [{elapsed}<{remaining}, {rate_fmt}]"
        )
        for step, batch in progress_bar:
            # Move batch to device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # (Optional) Gradient clipping used in Trainer (already in PrivacyEngine)
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()
            scheduler.step()

            training_loss_history.append(loss.item())

            # Update the postfix inline every 100 steps
            if (step + 1) % 100 == 0:
                avg_loss = np.mean(training_loss_history[-100:])
                progress_bar.set_postfix_str(f"Epoch {epoch+1} - Step {step+1} - Loss: {avg_loss:.4f}")
    
    cur_epsilon = privacy_engine.get_epsilon(delta=delta)
    print(f"Client {client_id} - ε = {cur_epsilon:.2f} (δ={delta})")
    
    # Set the position embeddings back to trainable
    model.distilbert.embeddings.position_embeddings.weight.requires_grad = True
 
    del privacy_engine



sample_rate = min(1.0, training_args.per_device_train_batch_size / len(subsampled_train_data))

# Create the dataloader
train_loader_dp = DataLoader(
    subsampled_train_data,
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True,
)
delta = 1 / len(train_loader_dp)

import opacus
epsilon = 10.0
sensitivity = 1.0
sigma = opacus.accountants.utils.get_noise_multiplier(
    target_epsilon=epsilon,
    # target_delta=cfg.delta,
    target_delta = 1 / len(train_loader_dp),
    sample_rate=sample_rate,
    epochs=int(training_args.num_train_epochs), 
    accountant='rdp',  
) 
print(f"Client {1} - Noise multiplier: {sigma}")

# Optimizer and scheduler
optimizer_dp = torch.optim.AdamW(
    model.parameters(),
    lr=training_args.learning_rate,
    betas=(training_args.adam_beta1, training_args.adam_beta2),
    eps=training_args.adam_epsilon,
    weight_decay=training_args.weight_decay,
)

model.train()
model.to("mps")
    
# Opacus privacy engine
privacy_engine = opacus.PrivacyEngine(accountant='rdp', secure_mode=False)
model, optimizer_dp, train_loader_dp = privacy_engine.make_private(
                        module=model,
                        optimizer=optimizer_dp,
                        data_loader=train_loader_dp,
                        noise_multiplier=sigma,
                        max_grad_norm=sensitivity,
                        poisson_sampling=False,
                    )
                
                
                
# 5. Train the model on the IN set
train_llm_with_opacus(
    model, 
    "mps", 
    subsampled_train_data,  
    training_args, 
    sigma,
    1 / len(train_loader_dp), 
    client_id=1
    )

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Client 1 - Noise multiplier: 0.35400390625


/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/opacus/privacy_engine.py:95: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
  0%|          | 0/24 -  [00:00<?, ?it/s]/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
100%|██████████| 24/24 -  [00:07<00:00,  3.34it/s]

Client 1 - ε = 10.09 (δ=0.041666666666666664)


In [4]:
# calculate the accuracy of the model on the test set
model.eval()
model.to("mps")
# pick only 500 samples
test_data = test_data.select(range(500))
test_loader = DataLoader(test_data, batch_size=16)
y_true = []
y_pred = []
losses = []
for batch in tqdm(test_loader):
    input_ids = batch["input_ids"].to("mps")
    attention_mask = batch["attention_mask"].to("mps")
    labels = batch["label"].to("mps")
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        y_pred.extend(logits.argmax(dim=-1).cpu().numpy())
        y_true.extend(labels.cpu().numpy())
        loss = cross_entropy(logits, labels)
        losses.append(loss.item())
        # print(f"Loss: {loss.item():.4f}")
        
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")
print(f"Test set F1: {f1:.4f}")
print(f"Test set loss: {np.mean(losses):.4f}")

100%|██████████| 32/32 [00:01<00:00, 23.81it/s]

Test set accuracy: 1.0000
Test set F1: 0.0000
Test set loss: 0.5972



/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
import math

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

train_loader = DataLoader(
        subsampled_train_data,
        batch_size=training_args.per_device_train_batch_size,
        shuffle=True,
)
sample_rate = min(1.0, training_args.per_device_train_batch_size / len(subsampled_train_data))

    
# Optimizer and scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_args.learning_rate,
    betas=(training_args.adam_beta1, training_args.adam_beta2),
    eps=training_args.adam_epsilon,
    weight_decay=training_args.weight_decay,
)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * training_args.num_train_epochs,
)

delta = 1 / len(train_loader)

import opacus
epsilon = 10.0
sensitivity = 1.0
sigma = opacus.accountants.utils.get_noise_multiplier(
    target_epsilon=epsilon,
    # target_delta=cfg.delta,
    target_delta = 1 / len(train_loader),
    sample_rate=sample_rate,
    epochs=int(training_args.num_train_epochs), 
    accountant='rdp',  
) 
print(f"Client {1} - Noise multiplier: {sigma}")


# def dp_sgd_train_loop(
#     model,
#     dataloader,
#     loss_fn,
#     max_grad_norm,
#     noise_multiplier,
#     lr,
#     device,
#     num_epochs=1
# ):
#     """
#     Manually performs DP-SGD with per-sample gradient clipping and noise addition.

#     Args:
#         model: Your nn.Module.
#         dataloader: DataLoader (or any iterable) giving full batches.
#                     Each batch can be large. We'll sub-divide each batch into microbatches.
#         loss_fn: The loss function.
#         max_grad_norm: C in the DP-SGD paper (the clipping bound).
#         noise_multiplier: the sigma in DP-SGD (noise = N(0, sigma^2 * C^2)).
#         lr: Learning rate.
#         device: 'cpu' or 'cuda'.
#         num_epochs: how many epochs we train.

#     Returns:
#         None (updates `model` in-place).
#     """

#     # Set the position embeddings back to trainable
#     model.distilbert.embeddings.position_embeddings.weight.requires_grad = False
 
#     model.to(device)
#     model.train()

#     for epoch in range(num_epochs):
#         for big_batch in tqdm(dataloader):
#             # Suppose big_batch itself has shape [B, ...]
#             # We will manually iterate over single-sample microbatches to replicate
#             # the Opacus per-sample gradient clipping.
#             # Move batch to device
#             x_batch = big_batch["input_ids"].to(device)
#             attention_mask = big_batch["attention_mask"].to(device)
#             y_batch = big_batch["label"].to(device)

#             # x_batch = big_batch[0].to(device)  # features
#             # y_batch = big_batch[1].to(device)  # labels
#             batch_size = x_batch.shape[0]

#             # 1) Initialize container for the per-sample grads
#             for p in model.parameters():
#                 # We'll store one gradient per sample
#                 p.accumulated_grads = []

#             # 2) For each sample in the batch, do forward/backward to get per-sample grad
#             for i in range(batch_size):
#                 x_i = x_batch[i].unsqueeze(0)  # shape [1, ...]
#                 y_i = y_batch[i].unsqueeze(0)  # shape [1, ...]
#                 a_i = attention_mask[i].unsqueeze(0)  # shape [1, ...]

#                 # Forward
#                 # Forward pass
#                 y_hat_i = model(input_ids=x_i, attention_mask=a_i, labels=y_i)
#                 loss_i = y_hat_i.loss
#                 # y_hat_i = model(x_i)
#                 # loss_i = loss_fn(y_hat_i, y_i)

#                 # Backward
#                 model.zero_grad()
#                 loss_i.backward()

#                 # 3) Clip each parameter's per-sample grad, then save it
#                 #    We do the clipping "per sample" or "per microbatch."
#                 #    We'll do a global norm clipping across all parameters,
#                 #    or you can do per-parameter clipping if you prefer.
#                 #    Example with global norm:
#                 #    (You could also do param-wise clip instead, as Opacus can do.)
#                 total_norm = 0.0
#                 # Compute global norm first
#                 for p in model.parameters():
#                     if p.grad is not None:
#                         total_norm += p.grad.data.norm(2).item()**2
#                 total_norm = math.sqrt(total_norm)

#                 clip_factor = min(1.0, max_grad_norm / (total_norm + 1e-6))

#                 # Now store clipped per-sample gradient
#                 for p in model.parameters():
#                     if p.grad is not None:
#                         grad_sample = p.grad.detach().clone()
#                         grad_sample.mul_(clip_factor)  # in-place scale
#                         p.accumulated_grads.append(grad_sample)
            
#             # 4) Aggregate all per-sample gradients for each parameter
#             for p in model.parameters():
#                 if len(p.accumulated_grads) == 0:
#                     continue
#                 # Stack them into shape [batch_size, *param.shape]
#                 stacked_grads = torch.stack(p.accumulated_grads, dim=0)
#                 # Average them (DP-SGD typically uses the *average* of the clipped grads)
#                 grad = stacked_grads.mean(dim=0)

#                 # 5) Add noise: N(0, sigma^2 * C^2 / batch_size^2).
#                 #    But frequently we just do noise with std = noise_multiplier * max_grad_norm / batch_size
#                 #    (depending on the exact formula you are using)
#                 noise = torch.normal(
#                     mean=0.0,
#                     std=(noise_multiplier * max_grad_norm) / batch_size,
#                     size=p.shape,
#                     device=p.device
#                 )
#                 grad += noise

#                 # Now store in p.grad so an optimizer can step
#                 p.grad = grad
            
#             # 6) Update parameters (SGD or Adam or anything).
#             #    We could create an optimizer, or do a manual update:
#             #    E.g., p = p - lr * p.grad
#             #    But typically you'd do an .step() with an optimizer:
#             # with torch.no_grad():
#             #     for p in model.parameters():
#             #         if p.grad is not None:
#             #             p -= lr * p.grad
#             optimizer.step()
#             scheduler.step()
#             optimizer.zero_grad()
            
#             # Zero out for next batch
#             model.zero_grad()
        
#         print(f"Epoch {epoch+1} finished.")
    
#     # Set the position embeddings back to trainable
#     model.distilbert.embeddings.position_embeddings.weight.requires_grad = True



def dp_sgd_train_loop(
    model,
    dataloader,
    optimizer,
    scheduler,
    max_grad_norm,       # C
    noise_multiplier,    # sigma
    device="cuda",
    num_epochs=1
):
    """
    Manually performs DP-SGD with EXACT same math as Opacus (poisson_sampling=False)
    for a single set of steps:
      1) Per-batch:
         - For each sample in the batch, compute grads
         - Globally clip to max_grad_norm
         - Then average per-sample grads across the batch
         - Add noise with std = sigma * C / batch_size
         - Apply optimizer step
    """
    # Example: freeze position embeddings if you do so in your code
    # (Must be consistent with your original Opacus code)
    model.distilbert.embeddings.position_embeddings.weight.requires_grad = False

    model.to(device)
    model.train()
    
    # We will assume the dataloader is the same as used with Opacus
    # and that you have set shuffle=True, drop_last=False, etc. consistently.
    
    for epoch in range(num_epochs):
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False):
            # Extract input tensors
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            # B = batch size
            B = input_ids.size(0)
            
            # 1) Zero out the container for per-sample grads
            for p in model.parameters():
                # We will store each sample's gradient in a list
                p.accumulated_grads = []

            # 2) Per-sample gradient computation loop
            for i in range(B):
                # Forward/backward on a single sample
                input_ids_i      = input_ids[i].unsqueeze(0)
                attention_mask_i = attention_mask[i].unsqueeze(0)
                labels_i         = labels[i].unsqueeze(0)

                # Forward
                outputs = model(
                    input_ids=input_ids_i,
                    attention_mask=attention_mask_i,
                    labels=labels_i
                )
                loss_i = outputs.loss

                # Clear grads
                model.zero_grad()
                # Backprop
                loss_i.backward()

                # We'll do global norm clipping, so first gather
                # the current per-sample grads in a flat sense
                grad_norm_sq = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        grad_norm_sq += p.grad.data.norm(2).item() ** 2
                grad_norm = math.sqrt(grad_norm_sq)

                # Clip factor
                clip_factor = min(1.0, max_grad_norm / (grad_norm + 1e-6))

                # Save each clipped gradient
                for p in model.parameters():
                    if p.grad is not None:
                        g = p.grad.detach().clone()
                        g.mul_(clip_factor)  # clip in-place
                        p.accumulated_grads.append(g)

            # 3) Now aggregate (average) all per-sample grads for each parameter
            for p in model.parameters():
                if len(p.accumulated_grads) == 0:
                    continue
                
                # Stack shape: [B, ...parameter_shape...]
                stacked = torch.stack(p.accumulated_grads, dim=0)
                # Average over the batch dimension
                grad = stacked.mean(dim=0)

                # 4) Add Gaussian noise: N(0, sigma^2 * C^2 / B^2)
                # => noise stddev = sigma*C / B
                noise = torch.normal(
                    mean=0.0,
                    std=(noise_multiplier * max_grad_norm) / B,
                    size=p.shape,
                    device=p.device,
                )
                grad += noise

                # Place in p.grad so we can do an optimizer step
                p.grad = grad

            # 5) Optimizer step
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            
            # 6) Zero out for next iteration
            optimizer.zero_grad()

        print(f"Epoch {epoch+1} done.")

    # Unfreeze if that’s what you do after training
    model.distilbert.embeddings.position_embeddings.weight.requires_grad = True

# dp_sgd_train_loop(
#     model,
#     train_loader,
#     cross_entropy,
#     max_grad_norm=1.0,
#     noise_multiplier=sigma,
#     lr=training_args.learning_rate,
#     device="mps",
#     num_epochs=1
# )

dp_sgd_train_loop(
    model,
    train_loader,
    optimizer,
    scheduler,
    max_grad_norm=1.0,       # C
    noise_multiplier=sigma,    # sigma
    device="mps",
    num_epochs=1,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Client 1 - Noise multiplier: 0.35400390625


Epoch 1 done.


In [3]:
# calculate the accuracy of the model on the test set
model.eval()
model.to("mps")
# pick only 500 samples
test_data = test_data.select(range(500))
test_loader = DataLoader(test_data, batch_size=16)
y_true = []
y_pred = []
losses = []
for batch in tqdm(test_loader):
    input_ids = batch["input_ids"].to("mps")
    attention_mask = batch["attention_mask"].to("mps")
    labels = batch["label"].to("mps")
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        y_pred.extend(logits.argmax(dim=-1).cpu().numpy())
        y_true.extend(labels.cpu().numpy())
        loss = cross_entropy(logits, labels)
        losses.append(loss.item())
        # print(f"Loss: {loss.item():.4f}")
        
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")
print(f"Test set F1: {f1:.4f}")
print(f"Test set loss: {np.mean(losses):.4f}")

100%|██████████| 32/32 [00:01<00:00, 27.86it/s]

Test set accuracy: 1.0000
Test set F1: 0.0000
Test set loss: 0.5966



/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
mine
Test set accuracy: 1.0000
Test set F1: 0.0000
Test set loss: 0.5966

opacus
Test set accuracy: 1.0000
Test set F1: 0.0000
Test set loss: 0.5972





.












.



In [2]:
# test with dp
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)   
    return {
        "accuracy": acc, 
        "f1": f1,
        }

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    overwrite_output_dir=True,
    # num_train_epochs=1, # Set desired number of epochs - Commented out to use max_steps
    # max_steps=100,  # Set desired number of training steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,             
    weight_decay=0.0,               
    # adam_beta1=0.9,                   
    # adam_beta2=0.999,                 
    # adam_epsilon=1e-8,   
    eval_strategy="no", #"epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    seed=42,
    # optim="adamw_hf"
)

# 5. Trainer initialization using only the IN set for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=subsampled_train_data,
    optimizers=(torch.optim.Adam(model.parameters(), lr=5e-5), None),
    # eval_dataset=test_data,  # Normal evaluation on the official test set (not pass it in FL)
    compute_metrics=compute_metrics,
)

In [25]:
data_loader = DataLoader(subsampled_train_data, batch_size=8, shuffle=True)

In [28]:
import opacus
privacy_engine = opacus.PrivacyEngine()
model.train()
model, optimizer_dp, dp_train_loader = privacy_engine.make_private(
                    module=model,
                    optimizer=trainer.optimizer,
                    data_loader=trainer.get_train_dataloader(),
                    noise_multiplier=sigma,
                    max_grad_norm=1.0,
                )

/Users/dariofenoglio/miniforge3/envs/eris/lib/python3.13/site-packages/opacus/privacy_engine.py:95: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [32]:
# test with dp
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)   
    return {
        "accuracy": acc, 
        "f1": f1,
        }

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    overwrite_output_dir=True,
    num_train_epochs=1, # Set desired number of epochs - Commented out to use max_steps
    # max_steps=100,  # Set desired number of training steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,             
    weight_decay=0.0,               
    # adam_beta1=0.9,                   
    # adam_beta2=0.999,                 
    # adam_epsilon=1e-8,   
    eval_strategy="no", #"epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    seed=42,
    # optim="adamw_hf"
)

# 5. Trainer initialization using only the IN set for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=subsampled_train_data,
    optimizers=(torch.optim.Adam(model.parameters(), lr=5e-5), None),
    # eval_dataset=test_data,  # Normal evaluation on the official test set (not pass it in FL)
    compute_metrics=compute_metrics,
)

sample_rate = min(1.0, training_args.per_device_train_batch_size / len(subsampled_train_data))

import opacus
sigma = opacus.accountants.utils.get_noise_multiplier(
    target_epsilon=10.0,
    target_delta=1e-5,
    sample_rate=sample_rate,
    epochs=int(training_args.num_train_epochs), 
    accountant='rdp',  
    ) 

privacy_engine = opacus.PrivacyEngine()
model.train()
model, optimizer_dp, dp_train_loader = privacy_engine.make_private(
                    module=model,
                    optimizer=trainer.optimizer,
                    data_loader=trainer.get_train_dataloader(),
                    noise_multiplier=sigma,
                    max_grad_norm=1.0,
                )


class DPTrainer(Trainer):
    def get_train_dataloader(self):
        # Return your DP-enabled data loader
        return dp_train_loader

# Then create and use the custom trainer:
trainer = DPTrainer(
    model=model,
    args=training_args,
    optimizers=(optimizer_dp, None),
    compute_metrics=compute_metrics,
)
trainer.train()

# Extract training loss from trainer's log history
training_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log]


TypeError: DistilBertForSequenceClassification.forward() got an unexpected keyword argument 'num_items_in_batch'

## Original code







In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)   
    return {
        "accuracy": acc, 
        "f1": f1,
        }

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    overwrite_output_dir=True,
    # num_train_epochs=None, # Set desired number of epochs - Commented out to use max_steps
    max_steps=1000,  # Set desired number of training steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,             
    weight_decay=0.0,               
    # adam_beta1=0.9,                   
    # adam_beta2=0.999,                 
    # adam_epsilon=1e-8,                
    eval_strategy="no", #"epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    seed=42
)

# 5. Trainer initialization using only the IN set for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=subsampled_train_data,
    # eval_dataset=test_data,  # Normal evaluation on the official test set (not pass it in FL)
    compute_metrics=compute_metrics,
)

# 6. Train the model
trainer.train()
# Extract training loss from trainer's log history
training_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log]

# Evaluate the model on the test dataset
eval_results = trainer.evaluate(eval_dataset=test_data)
accuracy = eval_results.get("eval_accuracy", None)
f1 = eval_results.get("eval_f1", None)
loss = eval_results.get("eval_loss", None)
print(f"Loss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1: {f1:.4f}")

# Inference
# y = trainer.predict(test_data) 

# Save the model
trainer.save_model("./distilbert-imdb")


Step,Training Loss
100,0.524600
200,0.489400
300,0.450800
400,0.409700
500,0.412700
600,0.403400
700,0.387000
800,0.381000
900,0.344400
1000,0.364200


Accuracy: 0.8599
Loss: 0.3361


In [4]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_scheduler  # For the linear LR scheduler
)
import numpy as np
import torch
import copy
import os
from torch.utils.data import DataLoader
from torch.nn.functional import cross_entropy
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# os.environ["CUDA_VISIBLE_DEVICES"] = "3"


# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

# The dataset["train"] subset has 25,000 samples. We'll split it into two halves.
full_train_data = dataset["train"]

# Shuffle the training set for a fair split
full_train_data = full_train_data.shuffle(seed=42)

# Split into IN set (first half) and OUT set (second half)
canary_frac = 0.5
n_canaries = int(len(full_train_data) * canary_frac)
canaries = full_train_data.select(range(0, n_canaries))
non_canaries = full_train_data.select(range(n_canaries, len(full_train_data)))
scores = np.zeros(n_canaries)

# We will train on in_set, and out_set will be used only for membership inference evaluation
test_data = dataset["test"]  # Standard test set (25,000 samples) for normal model evaluation if desired

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Freeze the word and position embeddings (cannot be optimized with opacus)
total_freezed = 0
for p in model.distilbert.embeddings.position_embeddings.parameters():
    p.requires_grad = False
    total_freezed += p.numel()
print(f"Freezed {total_freezed} parameters")

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
canaries = canaries.map(tokenize_function, batched=True)
non_canaries = non_canaries.map(tokenize_function, batched=True)

# (Optional) Tokenize standard test set
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
non_canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# subsample canaries & make new dataloader
true_in_out = torch.distributions.bernoulli.Bernoulli(torch.ones(n_canaries) * 0.5).sample()
true_in_out = true_in_out.numpy()
canaries_in_idx = torch.nonzero(torch.tensor(true_in_out))

# concatenate non_canaries data with samples from canaries with canaries_in_idx
subsampled_train_data = concatenate_datasets([non_canaries, canaries.select(canaries_in_idx)])
subsampled_train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Create DataLoaders
train_batch_size = 4
test_batch_size = 4

n_train = len(subsampled_train_data)
# Force a size that's a multiple of 4
truncate_to = (n_train // train_batch_size) * train_batch_size  
subsampled_train_data = subsampled_train_data.select(range(truncate_to))


train_dataloader = DataLoader(
    subsampled_train_data,
    batch_size=train_batch_size,
    shuffle=True,
)

test_dataloader = DataLoader(
    test_data,
    batch_size=test_batch_size,
    shuffle=False
)

# 5. Set up optimizer with same hyperparams as Trainer defaults
learning_rate = 5e-5
weight_decay = 0.0
betas = (0.9, 0.999)
adam_epsilon = 1e-8
max_grad_norm = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=betas,
    eps=adam_epsilon,
    weight_decay=weight_decay
)

# 6. Set up linear schedule (no warmup steps by default) to replicate Trainer's default
max_steps = 1000
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=max_steps
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else device)
model.to(device)

# 7. Training loop
global_step = 0
training_loss_history = []





# # make private
# sample_rate = min(1.0, train_dataloader.batch_size / len(subsampled_train_data))

# import opacus
# sigma = opacus.accountants.utils.get_noise_multiplier(
#     target_epsilon=500.0,
#     target_delta=1e-5,
#     sample_rate=sample_rate,
#     epochs=int(1), 
#     accountant='rdp',  
#     ) 

# privacy_engine = opacus.PrivacyEngine()
# model.train()
# model, optimizer, train_dataloader = privacy_engine.make_private(
#                     module=model,
#                     optimizer=optimizer,
#                     data_loader=train_dataloader,
#                     noise_multiplier=sigma,
#                     max_grad_norm=1.0,
#                     poisson_sampling=False
#                 )






# We will keep cycling over the train_dataloader until we hit max_steps
train_data_iter = iter(train_dataloader)

model.train()
while global_step < max_steps:
    try:
        batch = next(train_data_iter)
    except StopIteration:
        # Reinitialize the iterator if we've exhausted the dataloader
        train_data_iter = iter(train_dataloader)
        batch = next(train_data_iter)
    
    # Move batch to device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

    # Backward pass
    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping (like Trainer's max_grad_norm)
    # torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    optimizer.step()
    scheduler.step()

    training_loss_history.append(loss.item())

    if (global_step + 1) % 100 == 0:
        print(f"Step {global_step+1}/{max_steps} - Loss: {np.mean(training_loss_history):.4f}")

    global_step += 1

print("Finished training.")

# 8. Evaluate
model.eval()
all_preds = []
all_labels = []
eval_losses = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        eval_losses.append(loss.item())
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

eval_loss = np.mean(eval_losses)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Loss: {eval_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1: {f1:.4f}")

# # 9. Save the model
# save_path = "./distilbert-imdb"
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)
# print(f"Model saved to {save_path}")

# # Mirror the final training loss array as with Trainer's log history
# training_loss = training_loss_history

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Freezed 393216 parameters
Step 100/1000 - Loss: 0.6620
Step 200/1000 - Loss: 0.5671
Step 300/1000 - Loss: 0.5309
Step 400/1000 - Loss: 0.5159
Step 500/1000 - Loss: 0.4965
Step 600/1000 - Loss: 0.4957
Step 700/1000 - Loss: 0.4820
Step 800/1000 - Loss: 0.4749
Step 900/1000 - Loss: 0.4601
Step 1000/1000 - Loss: 0.4555
Finished training.
Test Loss: 0.3506
Test Accuracy: 0.8452
Test F1: 0.8408


In [4]:
# Create DataLoaders
train_batch_size = 4
test_batch_size = 4

n_train = len(subsampled_train_data)
# Force a size that's a multiple of 4
truncate_to = (n_train // train_batch_size) * train_batch_size  
subsampled_train_data = subsampled_train_data.select(range(truncate_to))


train_dataloader = DataLoader(
    subsampled_train_data,
    batch_size=train_batch_size,
    shuffle=True,
)

print(len(subsampled_train_data))
print(len(train_dataloader))
sample_rate = min(1.0, train_batch_size / len(subsampled_train_data))
print(sample_rate)
print(len(subsampled_train_data) // train_batch_size)   

18724
4681
0.00021362956633198035
4681


: 

: 

In [5]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_scheduler  # For the linear LR scheduler
)
import numpy as np
import torch
import copy
import os
from torch.utils.data import DataLoader
from torch.nn.functional import cross_entropy
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# os.environ["CUDA_VISIBLE_DEVICES"] = "3"


# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

# The dataset["train"] subset has 25,000 samples. We'll split it into two halves.
full_train_data = dataset["train"]

# Shuffle the training set for a fair split
full_train_data = full_train_data.shuffle(seed=42)

# Split into IN set (first half) and OUT set (second half)
canary_frac = 0.5
n_canaries = int(len(full_train_data) * canary_frac)
canaries = full_train_data.select(range(0, n_canaries))
non_canaries = full_train_data.select(range(n_canaries, len(full_train_data)))
scores = np.zeros(n_canaries)

# We will train on in_set, and out_set will be used only for membership inference evaluation
test_data = dataset["test"]  # Standard test set (25,000 samples) for normal model evaluation if desired

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# # Freeze the word and position embeddings (cannot be optimized with opacus)
# total_freezed = 0
# for p in model.distilbert.embeddings.position_embeddings.parameters():
#     p.requires_grad = False
#     total_freezed += p.numel()
# print(f"Freezed {total_freezed} parameters")

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
canaries = canaries.map(tokenize_function, batched=True)
non_canaries = non_canaries.map(tokenize_function, batched=True)

# (Optional) Tokenize standard test set
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
non_canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# subsample canaries & make new dataloader
true_in_out = torch.distributions.bernoulli.Bernoulli(torch.ones(n_canaries) * 0.5).sample()
true_in_out = true_in_out.numpy()
canaries_in_idx = torch.nonzero(torch.tensor(true_in_out))

# concatenate non_canaries data with samples from canaries with canaries_in_idx
subsampled_train_data = concatenate_datasets([non_canaries, canaries.select(canaries_in_idx)])
subsampled_train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Create DataLoaders
train_batch_size = 4
test_batch_size = 4

n_train = len(subsampled_train_data)
# Force a size that's a multiple of 4
truncate_to = (n_train // train_batch_size) * train_batch_size  
subsampled_train_data = subsampled_train_data.select(range(truncate_to))


train_dataloader = DataLoader(
    subsampled_train_data,
    batch_size=train_batch_size,
    shuffle=True,
)

test_dataloader = DataLoader(
    test_data,
    batch_size=test_batch_size,
    shuffle=False
)

# 5. Set up optimizer with same hyperparams as Trainer defaults
learning_rate = 5e-5
weight_decay = 0.0
betas = (0.9, 0.999)
adam_epsilon = 1e-8
max_grad_norm = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=betas,
    eps=adam_epsilon,
    weight_decay=weight_decay
)

# 6. Set up linear schedule (no warmup steps by default) to replicate Trainer's default
max_steps = 1000
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=max_steps
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else device)
model.to(device)

# 7. Training loop
global_step = 0
training_loss_history = []





# # make private
# sample_rate = min(1.0, train_dataloader.batch_size / len(subsampled_train_data))

# import opacus
# sigma = opacus.accountants.utils.get_noise_multiplier(
#     target_epsilon=500.0,
#     target_delta=1e-5,
#     sample_rate=sample_rate,
#     epochs=int(1), 
#     accountant='rdp',  
#     ) 

# privacy_engine = opacus.PrivacyEngine()
# model.train()
# model, optimizer, train_dataloader = privacy_engine.make_private(
#                     module=model,
#                     optimizer=optimizer,
#                     data_loader=train_dataloader,
#                     noise_multiplier=sigma,
#                     max_grad_norm=1.0,
#                     poisson_sampling=False
#                 )






# We will keep cycling over the train_dataloader until we hit max_steps
train_data_iter = iter(train_dataloader)

model.train()
while global_step < max_steps:
    try:
        batch = next(train_data_iter)
    except StopIteration:
        # Reinitialize the iterator if we've exhausted the dataloader
        train_data_iter = iter(train_dataloader)
        batch = next(train_data_iter)
    
    # Move batch to device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

    # Backward pass
    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping (like Trainer's max_grad_norm)
    # torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    optimizer.step()
    scheduler.step()

    training_loss_history.append(loss.item())

    if (global_step + 1) % 100 == 0:
        print(f"Step {global_step+1}/{max_steps} - Loss: {np.mean(training_loss_history):.4f}")

    global_step += 1

print("Finished training.")

# 8. Evaluate
model.eval()
all_preds = []
all_labels = []
eval_losses = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        eval_losses.append(loss.item())
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

eval_loss = np.mean(eval_losses)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Loss: {eval_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1: {f1:.4f}")

# # 9. Save the model
# save_path = "./distilbert-imdb"
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)
# print(f"Model saved to {save_path}")

# # Mirror the final training loss array as with Trainer's log history
# training_loss = training_loss_history

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 100/1000 - Loss: 0.6062
Step 200/1000 - Loss: 0.5408
Step 300/1000 - Loss: 0.5185
Step 400/1000 - Loss: 0.4922
Step 500/1000 - Loss: 0.4838
Step 600/1000 - Loss: 0.4632
Step 700/1000 - Loss: 0.4592
Step 800/1000 - Loss: 0.4363
Step 900/1000 - Loss: 0.4287
Step 1000/1000 - Loss: 0.4250
Finished training.
Test Loss: 0.3520
Test Accuracy: 0.8469
Test F1: 0.8457


In [30]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_scheduler  # For the linear LR scheduler
)
import opacus
from opacus.accountants.utils import get_noise_multiplier
import numpy as np
import torch
import copy
import os
from torch.utils.data import DataLoader
from torch.nn.functional import cross_entropy
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# os.environ["CUDA_VISIBLE_DEVICES"] = "3"


# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

# The dataset["train"] subset has 25,000 samples. We'll split it into two halves.
full_train_data = dataset["train"]

# Shuffle the training set for a fair split
full_train_data = full_train_data.shuffle(seed=42)

# Split into IN set (first half) and OUT set (second half)
canary_frac = 0.5
n_canaries = int(len(full_train_data) * canary_frac)
canaries = full_train_data.select(range(0, n_canaries))
non_canaries = full_train_data.select(range(n_canaries, len(full_train_data)))
scores = np.zeros(n_canaries)

# We will train on in_set, and out_set will be used only for membership inference evaluation
test_data = dataset["test"]  # Standard test set (25,000 samples) for normal model evaluation if desired

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
canaries = canaries.map(tokenize_function, batched=True)
non_canaries = non_canaries.map(tokenize_function, batched=True)

# (Optional) Tokenize standard test set
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
non_canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# subsample canaries & make new dataloader
true_in_out = torch.distributions.bernoulli.Bernoulli(torch.ones(n_canaries) * 0.5).sample()
true_in_out = true_in_out.numpy()
canaries_in_idx = torch.nonzero(torch.tensor(true_in_out))

# concatenate non_canaries data with samples from canaries with canaries_in_idx
subsampled_train_data = concatenate_datasets([non_canaries, canaries.select(canaries_in_idx)])
subsampled_train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Create DataLoaders
train_batch_size = 4
test_batch_size = 4
epochs = 1

n_train = len(subsampled_train_data)
# Force a size that's a multiple of 4
truncate_to = (n_train // train_batch_size) * train_batch_size  
subsampled_train_data = subsampled_train_data.select(range(truncate_to))

train_dataloader = DataLoader(
    subsampled_train_data,
    batch_size=train_batch_size,
    shuffle=True,
)

test_dataloader = DataLoader(
    test_data,
    batch_size=test_batch_size,
    shuffle=False
)

# 5. Set up optimizer with same hyperparams as Trainer defaults
learning_rate = 5e-5
weight_decay = 0.0
betas = (0.9, 0.999)
adam_epsilon = 1e-8
max_grad_norm = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=betas,
    eps=adam_epsilon,
    weight_decay=weight_decay
)

# 6. Set up linear schedule (no warmup steps by default) to replicate Trainer's default
max_steps = len(train_dataloader) * epochs
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=max_steps
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else device)
model.to(device)

# 7. Training loop
global_step = 0
training_loss_history = []

# # make private

# # Freeze the word and position embeddings (cannot be optimized with opacus)
model.distilbert.embeddings.position_embeddings.weight.requires_grad = False
# total_freezed = 0
# for p in model.distilbert.embeddings.position_embeddings.parameters():
#     p.requires_grad = False
#     total_freezed += p.numel()
# print(f"Freezed {total_freezed} parameters")

sample_rate = min(1.0, train_dataloader.batch_size / len(subsampled_train_data))

import opacus
sigma = opacus.accountants.utils.get_noise_multiplier(
    target_epsilon=10.0,
    target_delta=1e-5,
    sample_rate=sample_rate,
    epochs=int(epochs), 
    accountant='rdp',  
    ) 
print(f"sigma: {sigma}")
# sigma = 0.0

privacy_engine = opacus.PrivacyEngine(secure_mode=False)
model.train()
model, optimizer, train_dataloader = privacy_engine.make_private(
                    module=model,
                    optimizer=optimizer,
                    data_loader=train_dataloader,
                    noise_multiplier=sigma,
                    max_grad_norm=1.0,
                    poisson_sampling=False,
                )
# We will keep cycling over the train_dataloader until we hit max_steps
train_data_iter = iter(train_dataloader)

model.train()
# while global_step < max_steps:
#     try:
#         batch = next(train_data_iter)
#     except StopIteration:
#         # Reinitialize the iterator if we've exhausted the dataloader
#         train_data_iter = iter(train_dataloader)
#         batch = next(train_data_iter)
    
#     # Move batch to device
#     input_ids = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     labels = batch["label"].to(device)

#     # Forward pass
#     outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#     loss = outputs.loss

#     # Backward pass
#     optimizer.zero_grad()
#     loss.backward()

#     # Gradient clipping (like Trainer's max_grad_norm)
#     # torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

#     optimizer.step()
#     scheduler.step()

#     training_loss_history.append(loss.item())

#     if (global_step + 1) % 100 == 0:
#         print(f"Step {global_step+1}/{max_steps} - Loss: {np.mean(training_loss_history):.4f}")

#     global_step += 1

# print("Finished training.")

            
from tqdm import tqdm
training_loss_history = []
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    # Customize the bar format
    progress_bar = tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        bar_format="{percentage:3.0f}%|{bar:10}| {n_fmt}/{total_fmt} - {postfix} [{elapsed}<{remaining}, {rate_fmt}]"
    )
    for step, batch in progress_bar:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # (Optional) Gradient clipping (already in PrivacyEngine, if used)
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        scheduler.step()

        training_loss_history.append(loss.item())

        # Update the postfix inline every 100 steps
        if (step + 1) % 100 == 0:
            avg_loss = np.mean(training_loss_history[-100:])
            progress_bar.set_postfix_str(f"Step {step+1} - Loss: {avg_loss:.4f}")

print("Finished training.")
model.distilbert.embeddings.position_embeddings.weight.requires_grad = True


# 8. Evaluate
model.eval()
all_preds = []
all_labels = []
eval_losses = []

with torch.no_grad():
    for batch in tqdm(test_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        eval_losses.append(loss.item())
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

eval_loss = np.mean(eval_losses)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Loss: {eval_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1: {f1:.4f}")

# # 9. Save the model
# save_path = "./distilbert-imdb"
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)
# print(f"Model saved to {save_path}")

# # Mirror the final training loss array as with Trainer's log history
# training_loss = training_loss_history

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


sigma: 0.3498077392578125
Epoch 1/1


100%|██████████| 4692/4692 - , Step 4600 - Loss: 0.6892 [08:06<00:00,  9.64it/s]


Finished training.


100%|██████████| 6250/6250 [01:28<00:00, 70.87it/s]

Test Loss: 0.6895
Test Accuracy: 0.5008
Test F1: 0.6670


: 

In [21]:
# Evaluating MIA
prediction_output = trainer.predict(canaries)
logits = torch.tensor(prediction_output.predictions)
labels = torch.tensor(prediction_output.label_ids)

# Compute per-sample loss
losses = cross_entropy(logits, labels, reduction='none')

# Compute scores
scores = -losses.cpu().numpy()


In [22]:
def evaluate_privacy(scores):
    k_plus = 1/3
    k_min = 1/3
    ground_truth = copy.deepcopy(true_in_out)
    score_indices_sorted = np.argsort(scores)[::-1]
    classified_in = score_indices_sorted[:int(n_canaries * k_plus + 1)]
    classified_out = score_indices_sorted[int(n_canaries * (1 - k_min) + 1):]
    abstained = np.setdiff1d(score_indices_sorted, np.concatenate((classified_in, classified_out)))
    classification = np.zeros(n_canaries)
    classification[classified_in] = 1
    classification[abstained] = 2
    ground_truth[abstained] = 2
    W = ground_truth == classification
    num_correct = W.sum() - len(abstained)
    accuracy_mia = num_correct / (n_canaries - len(abstained))
    
    # tpr = np.sum(classification == true_in_out) / len(canaries_in_idx)
    # tnr = np.sum((1 - classification) == (1 - true_in_out)) / len(canaries_out_idx)
    # fpr = np.sum(classification == (1 - true_in_out)) / len(canaries_out_idx)
    # fnr = np.sum((1 - classification) == true_in_out) / len(canaries_in_idx)

    # compute empirical privacy estimate, which should be < epsilon w/ high probability
    # privacy_estimate = utils.get_eps_audit(
    #     m=n_canaries,
    #     r=n_canaries - len(abstained),
    #     v=num_correct,
    #     delta=cfg.delta,
    #     p=0.05)
    privacy_estimate = 0.0  
    
    # Kairouz privacy estimate from https://proceedings.mlr.press/v37/kairouz15.html
    # privacy_estimate = np.max([np.log(1 - cfg.delta - fpr) - np.log(fnr), 
                        # np.log(1 - cfg.delta - fnr) - np.log(fpr)])
                        
    return accuracy_mia, privacy_estimate

accuracy_mia, privacy_estimate = evaluate_privacy(scores)
print(f"Accuracy MIA: {accuracy_mia:.4f}")

Accuracy MIA: 0.5616


In [23]:
def get_parameters(model, config):
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

p = get_parameters(model, None)



## Data split

In [51]:
# 1. Load the IMDb dataset
dataset = load_dataset("imdb")

train_data = dataset["train"]
train_data = train_data.shuffle(seed=42)

test_data = dataset["test"]

# 2. Initialize tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocessing (tokenization) function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize IN and OUT sets
train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

# Set formats for PyTorch
train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Save the datasets 
train_data.save_to_disk("./train_data")
test_data.save_to_disk("./test_data")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [58]:
def IID_split_and_save_torch(dataset, num_parts, save_dir="./client_datasets", seed=1):
    """
    Splits a dataset into num_parts IID sub-datasets and saves each subset to disk.

    Parameters:
    - dataset (datasets.Dataset): The dataset to split.
    - num_parts (int): The number of parts to split the dataset into.
    - save_dir (str): Directory where the sub-datasets will be saved.
    - seed (int): Random seed for shuffling.

    Returns:
    - List of file paths where each subset is saved.
    """

    print(f"\nSplitting the dataset into {num_parts} IID parts...")

    # Create the directory if it doesn't exist
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Get the total number of samples
    total_samples = len(dataset)

    # Calculate the size of each subset
    subset_size = total_samples // num_parts

    # Shuffle the dataset indices
    torch.manual_seed(seed)
    indices = torch.randperm(total_samples).tolist()  # Convert to a Python list

    for i in range(num_parts):
        # Define the start and end indices for this subset
        start_idx = i * subset_size
        if i == num_parts - 1:
            end_idx = total_samples  # Include the remaining samples in the last subset
        else:
            end_idx = start_idx + subset_size

        # Select subset using Hugging Face `select`
        subset = dataset.select(indices[start_idx:end_idx])

        # Define the file path to save this subset
        subset_dir = os.path.join(save_dir, f"IID_data_client_{i+1}")
        print(f"Saving data client {i+1} to {subset_dir}...")

        # Save the subset
        subset.save_to_disk(subset_dir)

# Load datasets
train_data = load_from_disk("./train_data")
test_data = load_from_disk("./test_data")

# Split the training dataset into 5 clients
IID_split_and_save_torch(train_data, 5, seed=1, save_dir="./datasetsss")


Splitting the dataset into 5 IID parts...
Saving data client 1 to ./datasetsss/IID_data_client_1...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving data client 2 to ./datasetsss/IID_data_client_2...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving data client 3 to ./datasetsss/IID_data_client_3...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving data client 4 to ./datasetsss/IID_data_client_4...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving data client 5 to ./datasetsss/IID_data_client_5...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [65]:
# client 1
train_data_client_1 = load_from_disk("./datasetsss/IID_data_client_1")
# select the first 1000 samples for the sub
train_data_client_1 = train_data_client_1.select(range(0, 1000))

# Split into IN set (first half) and OUT set (second half)
canary_frac = 0.5
n_canaries = int(len(train_data_client_1) * canary_frac)
canaries = train_data_client_1.select(range(0, n_canaries))
non_canaries = train_data_client_1.select(range(n_canaries, len(train_data_client_1)))
scores = np.zeros(n_canaries)

canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
non_canaries.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# subsample canaries & make new dataloader
true_in_out = torch.distributions.bernoulli.Bernoulli(torch.ones(n_canaries) * 0.5).sample()
true_in_out = true_in_out.numpy()
canaries_in_idx = torch.nonzero(torch.tensor(true_in_out))

# concatenate non_canaries data with samples from canaries with canaries_in_idx
subsampled_train_data = concatenate_datasets([non_canaries, canaries.select(canaries_in_idx)])
subsampled_train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [66]:
# 2. Initialize model
model_name = "distilbert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
    }

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    overwrite_output_dir=True,
    num_train_epochs=10, # Set desired number of epochs - Commented out to use max_steps
    # max_steps=1000,  # Set desired number of training steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,             
    weight_decay=0.0,               
    # adam_beta1=0.9,                   
    # adam_beta2=0.999,                 
    # adam_epsilon=1e-8,                
    eval_strategy="no", #"epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    seed=42
)

# 5. Trainer initialization using only the IN set for training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=subsampled_train_data,
    # eval_dataset=test_data,  # Normal evaluation on the official test set (not pass it in FL)
    compute_metrics=compute_metrics,
)

# 6. Train the model
trainer.train()

# Evaluate the model on the test dataset
# eval_results = trainer.evaluate(eval_dataset=test_data)
# accuracy = eval_results.get("eval_accuracy", None)
# loss = eval_results.get("eval_loss", None)
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Loss: {loss:.4f}")

# Inference
# y = trainer.predict(test_data) 



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,0.546300
200,0.292400
300,0.175400
400,0.061600
500,0.008700
600,0.011000
700,0.000700
800,0.000300
900,0.000300


TrainOutput(global_step=970, training_loss=0.11307731342223502, metrics={'train_runtime': 113.5568, 'train_samples_per_second': 67.808, 'train_steps_per_second': 8.542, 'total_flos': 254999742412800.0, 'train_loss': 0.11307731342223502, 'epoch': 10.0})

In [67]:
# Evaluating MIA
prediction_output = trainer.predict(canaries)
logits = torch.tensor(prediction_output.predictions)
labels = torch.tensor(prediction_output.label_ids)

# Compute per-sample loss
losses = cross_entropy(logits, labels, reduction='none')

# Compute scores
scores = -losses.cpu().numpy()


In [68]:
def evaluate_privacy(scores):
    k_plus = 1/3
    k_min = 1/3
    ground_truth = copy.deepcopy(true_in_out)
    score_indices_sorted = np.argsort(scores)[::-1]
    classified_in = score_indices_sorted[:int(n_canaries * k_plus + 1)]
    classified_out = score_indices_sorted[int(n_canaries * (1 - k_min) + 1):]
    abstained = np.setdiff1d(score_indices_sorted, np.concatenate((classified_in, classified_out)))
    classification = np.zeros(n_canaries)
    classification[classified_in] = 1
    classification[abstained] = 2
    ground_truth[abstained] = 2
    W = ground_truth == classification
    num_correct = W.sum() - len(abstained)
    accuracy_mia = num_correct / (n_canaries - len(abstained))
    
    # tpr = np.sum(classification == true_in_out) / len(canaries_in_idx)
    # tnr = np.sum((1 - classification) == (1 - true_in_out)) / len(canaries_out_idx)
    # fpr = np.sum(classification == (1 - true_in_out)) / len(canaries_out_idx)
    # fnr = np.sum((1 - classification) == true_in_out) / len(canaries_in_idx)

    # compute empirical privacy estimate, which should be < epsilon w/ high probability
    # privacy_estimate = utils.get_eps_audit(
    #     m=n_canaries,
    #     r=n_canaries - len(abstained),
    #     v=num_correct,
    #     delta=cfg.delta,
    #     p=0.05)
    privacy_estimate = 0.0  
    
    # Kairouz privacy estimate from https://proceedings.mlr.press/v37/kairouz15.html
    # privacy_estimate = np.max([np.log(1 - cfg.delta - fpr) - np.log(fnr), 
                        # np.log(1 - cfg.delta - fnr) - np.log(fpr)])
                        
    return accuracy_mia, privacy_estimate

accuracy_mia, privacy_estimate = evaluate_privacy(scores)
print(f"Accuracy MIA: {accuracy_mia:.4f}")

Accuracy MIA: 0.7237


: 